# GPT-6 Luna Synthetic Data Generator — VS Code

**Run in this order:** 1. Install → 2. Enter API key → 3. Enter instructions → 4. Test 3 examples immediately (regular API) → 5. Test a Batch job with ONE request → 6. Submit your full Batch → 7. Retrieve full results.

**Team setup:** Create your own API key **inside the team's OpenAI project** (see section 2 of `docs/synthetic-data-generation.md`), not in a personal account. Never paste it into a notebook cell, a chat, a commit or a screenshot. This notebook asks for it in a hidden prompt (or reads `OPENAI_API_KEY` if you have set it). Open the notebook from inside the repository folder. Your finished dataset is saved to `data/raw/<your name>.jsonl`; intermediate files (test runs, batch IDs, raw requests) go to `data/work/`, which Git ignores. The full instructions are in `docs/synthetic-data-generation.md`.

**Important:** Running the test makes paid synchronous API calls. Setting `SUBMIT_BATCH = True` and running the submission cell sends a paid batch job; do it **only once for each dataset**. A batch runs asynchronously and can take up to **24 hours**; you do not need to keep VS Code open. [Official Batch API guide](https://developers.openai.com/api/docs/guides/batch).

**Local setup:** Install Python and the Python and Jupyter extensions in VS Code. Open this notebook in your project folder, select a local Python kernel (install `ipykernel` if prompted), and run the cells individually. No GPU is needed. Files are saved to the kernel’s current working directory, printed in Section 2. Do not use Run All to submit paid jobs accidentally.


## 1 — Install the official OpenAI Python SDK

**What it does:** Installs the OpenAI Python library used to call the API from VS Code.

**What to do:** **Just run the next cell once** in your selected Python environment (or after changing environments). No values to change.

**Code source:** `pip install openai` — [OpenAI Quickstart](https://developers.openai.com/api/docs/quickstart). **Adaptation:** `%pip` is Jupyter's notebook form.


In [ ]:
# OFFICIAL: https://developers.openai.com/api/docs/quickstart
# ADAPTED: notebook-specific %pip rather than terminal pip.

# EXPLANATION: Installs the OpenAI Python package in the selected Python environment.
# Run once in each new Python environment; this does not send an API request.
%pip install openai

## 2 — API key and Python imports

**What it does:** Loads OpenAI's client and standard Python tools. Creates the API client, using the `OPENAI_API_KEY` environment variable if set; otherwise it asks for your API key with hidden input. It also prints the local folder where generated files are saved.

**What to do:** Run the next cell and paste your key into the hidden prompt if asked. You create the key inside the team's OpenAI project, following the guide (section 2). It is billed to the team's account, so keep it private. Run this cell again after restarting the kernel. Never hardcode or commit your key to GitHub. **Edit the settings block in the next cell:** set `AUTHOR` to your own lowercase name with no spaces (it becomes your ID prefix, such as `sam-0001`), set `METHOD` to the generation method the team agreed on, and check that `MODEL` matches what the team lead told you. Everyone must use the same model. This cell also defines the JSONL writer and basic field validation used by all dataset outputs.

**Code sources:** `from openai import OpenAI` and `client = OpenAI()` are shown in [Text Generation](https://developers.openai.com/api/docs/guides/text). The `api_key=` argument is supported by the [OpenAI Python SDK](https://github.com/openai/openai-python). `json`, `os`, `getpass` and `pathlib` are standard Python libraries.


In [ ]:
# EXPLANATION: Import the OpenAI client and standard Python tools.
from openai import OpenAI  # OFFICIAL API: https://developers.openai.com/api/docs/guides/text
import json
import os
from getpass import getpass
from datetime import date
from pathlib import Path

# EXPLANATION: Read the key from your environment or enter it privately.
# The hidden prompt avoids storing the key in the notebook or GitHub.
api_key = (os.environ.get("OPENAI_API_KEY") or "").strip()
if not api_key:
    api_key = getpass("Enter your OpenAI API key: ").strip()
if not api_key:
    raise ValueError("An OpenAI API key is required.")
client = OpenAI(api_key=api_key)
del api_key

# ---- TEAM SETTINGS (EDIT THESE) ---------------------------------------------
AUTHOR = "yourname"          # lowercase, no spaces; becomes your ID prefix (e.g. sam-0001)
METHOD = "self_instruct"     # self_instruct | evol_instruct | magpie (the team's agreed method)
MODEL = "gpt-6-luna"         # the SAME model for everyone; confirm with the team lead
# ------------------------------------------------------------------------------

if AUTHOR in ("", "yourname") or any(ch.isspace() for ch in AUTHOR):
    raise ValueError("Set AUTHOR above to your own name (lowercase, no spaces), then run this cell again.")

# ADDED (Python): find the repository folder so files always land in the right place,
# whether the notebook is opened from the repo root or from notebooks/.
def find_repo_root(start):
    for folder in [start, *start.parents]:
        if (folder / "configs" / "generation.yaml").exists():
            return folder
    raise RuntimeError("Open this notebook from inside the repository folder.")

REPO_ROOT = find_repo_root(Path.cwd().resolve())
RAW_DIR = REPO_ROOT / "data" / "raw"    # the final dataset goes here
WORK_DIR = REPO_ROOT / "data" / "work"  # test runs, batch IDs and raw requests (ignored by Git)
RAW_DIR.mkdir(parents=True, exist_ok=True)
WORK_DIR.mkdir(parents=True, exist_ok=True)

# EXPLANATION: Existing open(...) calls save files in the current folder, which is now data/work/.
os.chdir(WORK_DIR)
print("Repository:", REPO_ROOT)
print("Work files:", WORK_DIR)
print("Final dataset:", RAW_DIR / f"{AUTHOR}.jsonl")

# Check the generated fields before saving, using the team's required field types.
def validate_examples(examples):
    for i, example in enumerate(examples, start=1):
        if not isinstance(example, dict):
            raise ValueError(f"Example {i} is not a JSON object.")
        for field in ("instruction", "output"):
            value = example.get(field)
            if not isinstance(value, str) or not value.strip():
                raise ValueError(f"Example {i}: {field} must be a non-empty string.")
        if "input" in example and not isinstance(example["input"], str):
            raise ValueError(f"Example {i}: input must be a string.")
        if "meta" in example and not isinstance(example["meta"], dict):
            raise ValueError(f"Example {i}: meta must be an object.")

# Write one JSON object per line; assign IDs locally rather than asking the model.
def save_jsonl(filename, examples, id_prefix):
    validate_examples(examples)
    with open(filename, "w", encoding="utf-8") as f:
        for i, example in enumerate(examples, start=1):
            record = {
                "id": f"{id_prefix}-{i:04d}",
                "instruction": example["instruction"],
                "input": example.get("input", ""),
                "output": example["output"],
            }
            # ADDED (Team): record how the data was made, using the fields the repo's checker expects.
            record["meta"] = {
                **(example.get("meta") or {}),
                "author": AUTHOR,
                "method": METHOD,
                "generator": MODEL,
                "route": "api-batch",
                "generated_on": date.today().isoformat(),
            }
            f.write(json.dumps(record, ensure_ascii=False, allow_nan=False) + "\n")


## 3 — Data generation instructions (EDIT THIS)

**What it does:** Stores your custom synthetic-data prompt in `DATA_INSTRUCTIONS`. The three-example test and both Batch jobs read the same instructions, so you don't have to retype them.

**What to do:** Replace `[REPLACE THIS WITH YOUR OWN SYNTHETIC DATA INSTRUCTIONS]` with the data you want the model to generate, keeping the required fields specified below unchanged. Then **run the next cell**. This is a normal Python string, not a separate OpenAI API feature.


In [ ]:
# ADDED (Python): ordinary string variable, NOT an OpenAI API feature.
# TEAM: start from the template for your method in docs/synthetic-data-generation.md (section 5).

# EXPLANATION: Write what ONE synthetic example should look like (content, fields, rules).
# The test and full batch reuse this exact string; there is no default data type.
DATA_INSTRUCTIONS = """
[REPLACE THIS WITH YOUR OWN SYNTHETIC DATA INSTRUCTIONS]

Each example must be a JSON object with these fields:
- "instruction": a non-empty string containing the task.
- "input": a string containing additional context, or "" if not needed.
- "output": a non-empty string containing the answer.
Do not generate IDs; Python assigns them when saving.
"""



#example:

#Generate synthetic customer reviews for a sentiment classification dataset.

#Each sample must contain:
#- "instruction": Classify the sentiment of the following customer review as positive, negative, or neutral.
#- "input": A realistic customer review of 10-30 words.
#- "output": Either "positive", "negative", or "neutral".

#Requirements:
#- Maintain an approximately balanced distribution of labels.
#- Cover diverse topics, including electronics, clothing, restaurants,
#  groceries, and online shopping.
#- Vary sentence structure, vocabulary, and writing style.
#- Avoid duplicate or overly similar reviews.
#- Ensure the sentiment label accurately matches the review.

#Example samples:

#{"instruction": "Classify the sentiment of the following customer review.", "input": "The headphones sound great and are comfortable to wear.", "output": "positive"}

#{"instruction": "Classify the sentiment of the following customer review.", "input": "The delivery arrived late and the packaging was damaged.", "output": "negative"}

#{"instruction": "Classify the sentiment of the following customer review.", "input": "The shirt arrived yesterday and was the size I ordered.", "output": "neutral"}

## 4 — Test: Generate 3 examples immediately

**What it does:** Sends **one regular Responses API request** to GPT-6 Luna, asks for three examples, checks the result, saves a JSONL dataset to `test_3_examples.jsonl`, in your local working directory.

**What to do:** After completing Sections 1–3, run this cell once. No value to change here. This is a paid API request; it is *not* a Batch API request. Inspect the saved JSONL to see whether your prompt produces the data you want. Adjust your prompt and test until you're satisfied with the output. 

**API sources:** `client.responses.create(model=..., input=...)` and `response.output_text` — [Text Generation → simple Python prompt](https://developers.openai.com/api/docs/guides/text#generate-text-from-a-simple-prompt). Model ID — [GPT-6 Luna](https://developers.openai.com/api/docs/models/gpt-6-luna). `text={"format": {"type": "json_object"}}` — [OpenAI JSON Mode](https://developers.openai.com/api/docs/guides/structured-outputs#json-mode). JSON mode requests valid JSON, not an exact count.

**Other code:** The instructions requesting three items, `json.loads()` / `json.dumps()`, count and field checks, are Python additions to turn the model's text into a real file. [Python JSON docs](https://docs.python.org/3/library/json.html).


In [ ]:
# ADDED (Python): include a quantity + an internal JSON wrapper in YOUR prompt.
# OpenAI's input= accepts a string; the API has no built-in "number of dataset examples" setting.
# JSON mode requires that your prompt explicitly request JSON.

# OFFICIAL API CALL pattern: https://developers.openai.com/api/docs/guides/text

if "[REPLACE THIS" in DATA_INSTRUCTIONS:
    raise ValueError("Replace the placeholder in DATA_INSTRUCTIONS (Section 3) before running this cell.")

# EXPLANATION: Send ONE normal (non-Batch) API request and wait for its answer.
# The model returns JSON-formatted TEXT, not a file.
response = client.responses.create(
    model=MODEL,  # ADAPTED: https://developers.openai.com/api/docs/models/gpt-6-luna

    # EXPLANATION: input= is the actual prompt sent to the model.
    # The f-string adds a count (3) and asks for an "examples" JSON array.
    input=f"""{DATA_INSTRUCTIONS}

Generate exactly 3 examples.
Return a JSON object with one key, "examples", containing an array of exactly 3 examples.
""",  # ADAPTED: replace the bedtime-story prompt with your instructions + count.
    text={"format": {"type": "json_object"}},  # OFFICIAL JSON MODE: https://developers.openai.com/api/docs/guides/structured-outputs#json-mode
)

# ADDED (Python): parse JSON-formatted text, then extract the examples array.
# response.output_text is OFFICIAL (Text Generation example).

# EXPLANATION: Do not use the response if the generation was interrupted or incomplete.
if response.status != "completed":
    raise RuntimeError(f"Response did not finish: {response.status}")

# EXPLANATION: json.loads turns the JSON text into Python data; ["examples"] removes the
# outer {"examples": [...]} wrapper and keeps the list of examples.
examples = json.loads(response.output_text)["examples"]

# EXPLANATION: Check that the model actually returned a list containing three records.
# JSON mode alone does NOT guarantee the number of generated records.
if not isinstance(examples, list) or len(examples) != 3:
    raise ValueError("The model did not return exactly 3 examples.")

# ADDED (Python): validate fields and save one JSON object per line.
save_jsonl("test_3_examples.jsonl", examples, f"{AUTHOR}-test")
print("Saved locally:", Path("test_3_examples.jsonl").resolve())


## 5A — Batch API test: submit ONE request

**What it does:** Creates `test_batch_input.jsonl` with **one request asking for three examples**, uploads that input file to OpenAI, and submits a **test Batch job**. It prints the Batch ID and saves `test_batch_id.txt` locally so you can find the results later.

**What to do:** After trying Section 4, change `SUBMIT_TEST_BATCH = False` to **`True`** and run this cell **once**. Save `test_batch_id.txt`. You can close VS Code while OpenAI processes the job; use Section 5B to retrieve its result. **Do not rerun the submission cell** to check progress, because doing so starts another paid job.

**API sources:** JSONL request fields (`custom_id`, `method`, `url`, `body`) — [Prepare input file](https://developers.openai.com/api/docs/guides/batch#1-prepare-your-batch-file). `client.files.create(..., purpose="batch")` — [Upload batch input](https://developers.openai.com/api/docs/guides/batch#2-upload-your-batch-input-file). `client.batches.create(...)` — [Create the batch](https://developers.openai.com/api/docs/guides/batch#3-create-the-batch). The JSON-mode option comes from [Structured Outputs → JSON mode](https://developers.openai.com/api/docs/guides/structured-outputs#json-mode). Generating the one-line file and saving the Batch ID are additional Python code.


In [ ]:
# ADDED (Python): toggle to prevent accidentally submitting a paid test job.
SUBMIT_TEST_BATCH = False  # Set to True and run ONCE to test one Batch API request.

# EXPLANATION: Nothing is submitted while the switch above is False.
# Setting it True makes the entire indented block run ONCE each time you execute this cell.
if SUBMIT_TEST_BATCH:
    if "[REPLACE THIS" in DATA_INSTRUCTIONS:
        raise ValueError("Replace the placeholder in DATA_INSTRUCTIONS (Section 3) before running this cell.")

    # ADDED (Python): create a file containing ONE request, not 500 requests.
    # OFFICIAL JSONL request fields: https://developers.openai.com/api/docs/guides/batch#1-prepare-your-batch-file

    # EXPLANATION: Build one request envelope: custom_id identifies it; body contains the
    # ordinary Responses API model, prompt, and JSON-mode settings.
    request = {
        "custom_id": "test-request-1",  # OFFICIAL field; test identifier chosen by me.
        "method": "POST",              # OFFICIAL Batch request field.
        "url": "/v1/responses",         # OFFICIAL supported Batch endpoint.
        "body": {
            "model": MODEL, # Model ID: https://developers.openai.com/api/docs/models/gpt-6-luna

            # EXPLANATION: Reuse the SAME user instructions as the immediate test in Section 4.
            "input": f"""{DATA_INSTRUCTIONS}

Generate exactly 3 examples.
Return a JSON object with one key, "examples", containing an array of exactly 3 examples.
""",  # ADDED: user prompt + request for 3 examples (same as Section 4).
            "text": {"format": {"type": "json_object"}},  # OFFICIAL JSON mode.
        },
    }

    # EXPLANATION: Create a JSONL INPUT file: one JSON object per line.
    # Here the file contains exactly ONE line (one API request).
    with open("test_batch_input.jsonl", "w", encoding="utf-8") as f:

        # EXPLANATION: json.dumps serializes the request to text; the newline ends this JSONL record.
        # This is the request file, NOT the final generated dataset.
        f.write(json.dumps(request, ensure_ascii=False) + "\n")  # PYTHON JSONL writer.

    # OFFICIAL API: https://developers.openai.com/api/docs/guides/batch#2-upload-your-batch-input-file

    # EXPLANATION: Upload the local JSONL file to OpenAI and get back its input-file ID.
    # purpose="batch" tells OpenAI that the uploaded file is for Batch API.
    test_batch_input_file = client.files.create(
        file=open("test_batch_input.jsonl", "rb"), purpose="batch"
    )

    # OFFICIAL API: https://developers.openai.com/api/docs/guides/batch#3-create-the-batch
    # ADAPTED: /v1/responses rather than guide's chat/completions Python example.

    # EXPLANATION: Start the Batch job using that uploaded file ID.
    # The call returns a Batch ID immediately; OpenAI processes it separately.
    test_batch = client.batches.create(
        input_file_id=test_batch_input_file.id,
        endpoint="/v1/responses",
        completion_window="24h",
    )

    # ADDED (Python): record ID so it survives a kernel shutdown.

    # EXPLANATION: Save only the Batch ID in a tiny text file, so you can return later.
    # The ID refers to the job; it is NOT the generated examples.
    with open("test_batch_id.txt", "w", encoding="utf-8") as f:
        f.write(test_batch.id)
    print("TEST Batch submitted. ID:", test_batch.id)
    print("Save this TEST ID for the retrieval cell immediately below.")

    # EXPLANATION: Print the location of your saved job ID for the next retrieval section.
    print("Saved locally:", Path("test_batch_id.txt").resolve())
else:
    print("Test submission disabled. Set SUBMIT_TEST_BATCH = True to submit one request.")


### 5B — Check and retrieve your TEST Batch result

**What it does:** Asks OpenAI for the test Batch's status. If the job is complete, retrieves its raw `.jsonl` output, extracts the three examples, saves `test_batch_3_examples.jsonl`, in your local working directory.

**What to do:** Set `TEST_BATCH_ID = "PASTE_YOUR_TEST_BATCH_ID_HERE"` to the ID from Section 5A (or `test_batch_id.txt`), then **run the next cell**. If the status isn't `completed`, return later and **rerun only this retrieval cell**, not the submission cell. After a kernel restart, run Sections 1 and 2 first to reconnect to the API.

**API sources:** `client.batches.retrieve(...)` — [Check batch status](https://developers.openai.com/api/docs/guides/batch#4-check-the-status-of-a-batch); `client.files.content(...)` — [Retrieve output](https://developers.openai.com/api/docs/guides/batch#5-retrieve-the-results). Parsing the JSONL and saving a dataset `.jsonl` file are Python additions.


In [ ]:
# ADDED: paste the TEST Batch ID, e.g. from test_batch_id.txt.

# EXPLANATION: Paste the ID shown by Section 5A, not the model name or file name.
TEST_BATCH_ID = "PASTE_YOUR_TEST_BATCH_ID_HERE"

# OFFICIAL API: https://developers.openai.com/api/docs/guides/batch#4-check-the-status-of-a-batch

# EXPLANATION: Ask OpenAI for the status of the ALREADY submitted job.
# This does not start another generation job.
batch_test_status = client.batches.retrieve(TEST_BATCH_ID)
print("TEST Batch status:", batch_test_status.status)

# EXPLANATION: Only try downloading generated data after OpenAI reports completion.
# Otherwise you can come back and rerun this retrieval cell.
if batch_test_status.status == "completed":
    if not batch_test_status.output_file_id:
        raise RuntimeError("Test Batch completed without a successful output file.")

    # OFFICIAL API: https://developers.openai.com/api/docs/guides/batch#5-retrieve-the-results

    # EXPLANATION: Download the Batch OUTPUT from OpenAI using its output_file_id.
    # Its contents are JSONL lines containing API results, not a final JSON array.
    test_file_response = client.files.content(batch_test_status.output_file_id)

    # ADDED (Python): Batch returns JSONL envelopes, not bare model text.

    # EXPLANATION: Split the JSONL file into lines and convert each line into a Python dict.
    # The test should produce one result line because we submitted one request.
    test_rows = [json.loads(line) for line in test_file_response.text.splitlines() if line.strip()]
    if len(test_rows) != 1:
        raise ValueError(f"Expected 1 Batch result, received {len(test_rows)}.")
    row = test_rows[0]

    # EXPLANATION: Make sure that this one API request succeeded (HTTP status 200).
    if row.get("error") or not row.get("response") or row["response"]["status_code"] != 200:
        raise ValueError(f"Test request failed: {row.get('error') or row.get('response')}")

    # EXPLANATION: The response envelope contains metadata; "body" holds the model response.
    body = row["response"]["body"]
    if body.get("status") != "completed":
        raise ValueError(f"Incomplete model response: {body.get('status')}")

    # ADDED (Python): the raw Responses API result contains text in output[].content[].text.
    # Source for response structure: https://developers.openai.com/api/reference/resources/responses/methods/create

    # EXPLANATION: Locate the text generated by the model inside the Responses API output structure.
    # A Batch output row does not have the convenient response.output_text property.
    text_parts = [
        content["text"]
        for item in body.get("output", [])
        for content in item.get("content", [])
        if content.get("type") == "output_text"
    ]

    # EXPLANATION: Join the text fragments, parse the JSON string, and pull out "examples".
    # The result is a Python list that will be written as JSONL.
    test_examples = json.loads("".join(text_parts))["examples"]
    if not isinstance(test_examples, list) or len(test_examples) != 3:
        raise ValueError("Test Batch did not return exactly 3 examples.")

    # ADDED (Python): validate fields and save one JSON object per line.
    save_jsonl("test_batch_3_examples.jsonl", test_examples, f"{AUTHOR}-batch-test")
    print("Saved locally:", Path("test_batch_3_examples.jsonl").resolve())
else:
    print("TEST Batch isn't finished. Return later and rerun this retrieval cell.")


## 6 — Create and submit the FULL Batch job

**What it does:** Splits your requested total into multiple prompts, creates a `.jsonl` input file containing all the requests, uploads it to OpenAI, and submits the full Batch job. By default, `1_000` examples ÷ `20` per request means **50 requests**. Keep `TOTAL_EXAMPLES` equal to `num_examples` in `configs/generation.yaml`. It prints the full Batch ID and saves `batch_id.txt` locally.

**What to do:** First check that your prompt works in the test cells. If needed, change `TOTAL_EXAMPLES` and `EXAMPLES_PER_REQUEST`. Then change `SUBMIT_BATCH = False` to **`True`** and **run the next cell once**. Save `batch_id.txt`. **Do not rerun to check progress:** every submission creates another billable job. Your finished dataset is retrieved in Section 7, not here.

**API sources:** JSONL request layout — [Prepare batch input](https://developers.openai.com/api/docs/guides/batch#1-prepare-your-batch-file); `client.files.create(..., purpose="batch")` — [Upload input](https://developers.openai.com/api/docs/guides/batch#2-upload-your-batch-input-file); `client.batches.create(...)` — [Submit batch](https://developers.openai.com/api/docs/guides/batch#3-create-the-batch). Splitting the dataset into requests and creating the local ID file are Python additions.


In [ ]:
# ADDED (Python): these are dataset settings, NOT API parameters.

# EXPLANATION: Choose how many records you want in total.
# TEAM: keep this equal to num_examples in configs/generation.yaml.
TOTAL_EXAMPLES = 1_000

# EXPLANATION: Choose how many records EACH API request asks the model to generate.
# 1,000 total / 20 per request = 50 requests.
EXAMPLES_PER_REQUEST = 20

# EXPLANATION: Keep False while editing. Turn True and run ONCE when ready to pay for the job.
SUBMIT_BATCH = False  # Change to True to submit ONE paid job.

if SUBMIT_BATCH:
    if "[REPLACE THIS" in DATA_INSTRUCTIONS:
        raise ValueError("Replace the placeholder in DATA_INSTRUCTIONS (Section 3) before running this cell.")


    # EXPLANATION: Basic check: neither requested count can be zero or negative.
    if TOTAL_EXAMPLES < 1 or EXAMPLES_PER_REQUEST < 1:
        raise ValueError("Counts must be positive.")

    # ADDED (Python): create the .jsonl input file required by the Batch API.
    # OFFICIAL shape of each line: https://developers.openai.com/api/docs/guides/batch#1-prepare-your-batch-file
    with open("batch_input.jsonl", "w", encoding="utf-8") as f:

        # EXPLANATION: Make a request for each consecutive group of examples.
        # "start" is the offset (0, 20, 40, ...) used to identify each group.
        for start in range(0, TOTAL_EXAMPLES, EXAMPLES_PER_REQUEST):

            # EXPLANATION: Ask for at most EXAMPLES_PER_REQUEST, or fewer for the final leftover group.
            # Example: 105 total with 20 each -> last request asks for 5.
            count = min(EXAMPLES_PER_REQUEST, TOTAL_EXAMPLES - start)

            # EXPLANATION: Build the official Batch request envelope for this group.
            # custom_id encodes its position and expected size for later matching.
            request = {
                "custom_id": f"request-{start:06d}-{count:03d}",  # OFFICIAL field; ADDED value for matching results.
                "method": "POST",       # OFFICIAL Batch request field.
                "url": "/v1/responses", # ADAPTED from chat/completions; supported in Batch docs.
                "body": {
                    "model": MODEL,  # OFFICIAL model ID: https://developers.openai.com/api/docs/models/gpt-6-luna

                    # EXPLANATION: Each prompt combines your unchanged data instructions and this group's count.
                    "input": f"""{DATA_INSTRUCTIONS}

Generate exactly {count} examples.
Return a JSON object with one key, "examples", containing an array of exactly {count} examples.
""",  # ADAPTED: your instructions + the number for THIS request.
                    "text": {"format": {"type": "json_object"}},  # OFFICIAL JSON mode.
                },
            }

            # EXPLANATION: Write one request per line to batch_input.jsonl (the Batch API input file).
            # After the loop the file contains all requests, not generated responses.
            f.write(json.dumps(request, ensure_ascii=False) + "\n")

    # OFFICIAL API (Python): https://developers.openai.com/api/docs/guides/batch#2-upload-your-batch-input-file

    # EXPLANATION: Upload the finished JSONL request list; the returned object contains its file ID.
    batch_input_file = client.files.create(
        file=open("batch_input.jsonl", "rb"), purpose="batch"
    )

    # OFFICIAL API (Python): https://developers.openai.com/api/docs/guides/batch#3-create-the-batch
    # ADAPTED: endpoint="/v1/responses" instead of the docs' chat/completions example.

    # EXPLANATION: Submit that uploaded list as ONE Batch job for the /v1/responses endpoint.
    # 24h is the allowed processing window, not a guaranteed wait time.
    batch = client.batches.create(
        input_file_id=batch_input_file.id,
        endpoint="/v1/responses",
        completion_window="24h",
    )

    # ADDED (Python): save ID so you can retrieve results even after the kernel restarts.

    # EXPLANATION: Persist this full-job Batch ID. Use it in Section 7 to check and retrieve.
    # Do not rerun the submission cell just to check on a job.
    with open("batch_id.txt", "w", encoding="utf-8") as f:
        f.write(batch.id)
    print("Batch submitted. Batch ID:", batch.id)
    print("Copy this FULL Batch ID into Section 7 after the batch finishes.")

    # EXPLANATION: Print the location of the locally saved job ID.
    print("Saved locally:", Path("batch_id.txt").resolve())
else:
    print("Submission disabled. Set SUBMIT_BATCH = True when ready.")

## 7 — Check the FULL Batch and save your final `.jsonl` dataset

**What it does:** Checks the full Batch's status, retrieves the completed `.jsonl` output, extracts each request's examples, puts them in their original request order, and saves your final dataset to **`data/raw/<your name>.jsonl`**. If results are missing or invalid, it saves **`<your name>_PARTIAL.jsonl`** in `data/work/` instead (so an incomplete file never lands in `data/raw/`) and may also save `batch_errors.jsonl` there.

**What to do:** Paste the saved full ID from Section 6 (or `batch_id.txt`) into `BATCH_ID`, then **run the next cell**. If the job is unfinished, come back later and **rerun this retrieval cell only**. If the kernel has restarted, run Sections 1 and 2, then Section 6 with `SUBMIT_BATCH = False` to restore `TOTAL_EXAMPLES` and `EXAMPLES_PER_REQUEST`. **Never resubmit Section 6 just to retrieve results.**

**API sources:** `client.batches.retrieve(...)` — [Check status](https://developers.openai.com/api/docs/guides/batch#4-check-the-status-of-a-batch); `client.files.content(batch.output_file_id)` — [Retrieve results](https://developers.openai.com/api/docs/guides/batch#5-retrieve-the-results). Parsing JSONL envelopes, checking example counts, reordering by `custom_id`, and creating a final JSONL dataset are additional Python code.

**Important:** Batch results may be returned out of order. This notebook does not automatically retry failed requests. Retrieve and save the output after completion, because OpenAI's result files are not permanent.

**Repository checks:** From the repository root, run `python -m src.validate data/raw/<your name>.jsonl` and then `python -m src.check_data data/raw/<your name>.jsonl`. The first checks the format and unique IDs; the second runs the team's full standard check (duplicates, refusals, leftovers). Neither judges content quality.


In [ ]:
# ADDED: paste the FULL Batch ID printed by Section 6 (or read batch_id.txt).

# EXPLANATION: Paste the Batch ID printed by Section 6 (NOT the test Batch ID).
BATCH_ID = "PASTE_YOUR_BATCH_ID_HERE"

# OFFICIAL API: https://developers.openai.com/api/docs/guides/batch#4-check-the-status-of-a-batch

# EXPLANATION: Fetch the current status of the already-submitted full job from OpenAI.
# Checking status is NOT another dataset-generation request.
batch = client.batches.retrieve(BATCH_ID)
print("Batch status:", batch.status)

# EXPLANATION: Process the output only after the Batch job finishes.
# If it is still running, the else branch tells you to return later.
if batch.status == "completed":
    if not batch.output_file_id:
        raise RuntimeError("Batch completed but contains no successful output file.")

    # OFFICIAL API: https://developers.openai.com/api/docs/guides/batch#5-retrieve-the-results

    # EXPLANATION: Retrieve OpenAI's raw JSONL result file using the output file ID.
    file_response = client.files.content(batch.output_file_id)

    # ADDED (Python): each Batch output line is a JSON object with a response envelope.
    # Batch docs warn that the order of returned lines is not guaranteed.

    # EXPLANATION: Collect successful groups by their original starting offsets.
    # Batch results may arrive out of order, so sorting them later matters.
    results_by_start = {}

    # EXPLANATION: Remember failed or malformed group responses rather than silently dropping them.
    problems = []

    # EXPLANATION: Read each JSONL output line as a separate finished Batch API request.
    # The outer JSON object includes custom_id, response metadata, and body.
    for line in file_response.text.splitlines():
        row = json.loads(line)
        request_id = row["custom_id"]

        # EXPLANATION: Read the group's starting offset from custom_id, e.g. request-000020-020 -> 20.
        start = int(request_id.split("-")[1])  # From the full-Batch custom_id format in Section 6.

        # EXPLANATION: Read how many examples that group SHOULD contain from its custom_id.
        expected = int(request_id.split("-")[2])

        try:

            # EXPLANATION: Reject requests that the API reports as unsuccessful.
            if row.get("error") or row["response"]["status_code"] != 200:
                raise ValueError("API request failed")
            body = row["response"]["body"]
            if body.get("status") != "completed":
                raise ValueError(f"Incomplete response: {body.get('status')}")

            # ADDED (Python): raw Batch Responses data holds text in output[].content[].text.
            # https://developers.openai.com/api/reference/resources/responses/methods/create

            # EXPLANATION: Find the generated JSON text within this request's Responses API output.
            # The raw Batch file nests text under output -> content -> text.
            text_parts = [
                content["text"]
                for item in body.get("output", [])
                for content in item.get("content", [])
                if content.get("type") == "output_text"
            ]

            # EXPLANATION: Convert the JSON text to Python data and remove the {"examples": ...} wrapper.
            examples = json.loads("".join(text_parts))["examples"]
            if not isinstance(examples, list) or len(examples) != expected:
                raise ValueError(f"Expected {expected} examples")

            # EXPLANATION: Validate the generated record fields before accepting the group.
            validate_examples(examples)

            # EXPLANATION: Store this valid group under its offset to reconstruct request order.
            results_by_start[start] = examples
        except (KeyError, TypeError, ValueError, json.JSONDecodeError) as exc:

            # EXPLANATION: Record the request ID and failure reason for a later warning.
            problems.append(f"{request_id}: {exc}")

    # ADDED (Python): flatten request results in original order, then save JSONL.

    # EXPLANATION: Sort groups by starting position, then combine their lists into one big list.
    # all_examples exists in Python memory until it is written to JSONL below.
    all_examples = [example for start in sorted(results_by_start) for example in results_by_start[start]]

    # EXPLANATION: Calculate how many request groups were expected, rounding UP when needed.
    expected_requests = (TOTAL_EXAMPLES + EXAMPLES_PER_REQUEST - 1) // EXAMPLES_PER_REQUEST

    # EXPLANATION: Only label the dataset complete if counts match AND no problems were found.
    complete = (len(all_examples) == TOTAL_EXAMPLES and
                len(results_by_start) == expected_requests and not problems and
                not batch.error_file_id)

    # EXPLANATION: Use a distinct PARTIAL filename if examples are missing or requests failed.
    # Never quietly label an incomplete dataset as a successful full run.
    filename = RAW_DIR / f"{AUTHOR}.jsonl" if complete else WORK_DIR / f"{AUTHOR}_PARTIAL.jsonl"

    # EXPLANATION: Assign unique IDs and save one example per line as JSONL.
    save_jsonl(filename, all_examples, AUTHOR)
    print(f"Saved {len(all_examples)} examples to {filename}")
    if problems:
        print("Problematic results (first 5):", problems[:5])

    # OFFICIAL API: Batch errors can be retrieved through error_file_id.

    # EXPLANATION: If OpenAI has a separate error file, also download it for investigation.
    if batch.error_file_id:
        error_text = client.files.content(batch.error_file_id).text
        with open("batch_errors.jsonl", "w", encoding="utf-8") as f:
            f.write(error_text)
        print("Saved locally:", Path("batch_errors.jsonl").resolve())
        print("Some requests failed; saved batch_errors.jsonl locally.")

    # EXPLANATION: Print the location of the completed or partial local JSONL dataset.
    print("Saved locally:", Path(filename).resolve())
else:
    print("No final file yet. Run this cell again after the Batch finishes.")

## Source table

| Exact method / code | Official source | How it was used here |
|---|---|---|
| `from openai import OpenAI`; `client.responses.create(model=..., input=...)`; `response.output_text` | [Text Generation → simple Python prompt](https://developers.openai.com/api/docs/guides/text#generate-text-from-a-simple-prompt) | **Same API call**, with Luna and custom prompt substituted; test only |
| `OpenAI(api_key=...)` | [openai-python SDK](https://github.com/openai/openai-python) | Environment variable or privately entered key |
| `gpt-6-luna` (the `MODEL` setting) | [GPT-6 Luna model](https://developers.openai.com/api/docs/models/gpt-6-luna) | Different model from the guide's Astra example; the team can change `MODEL` in Section 2 |
| `text={"format": {"type": "json_object"}}` | [Structured Outputs → JSON Mode](https://developers.openai.com/api/docs/guides/structured-outputs#json-mode) | Requests JSON; it does **not** guarantee the number of examples |
| JSONL line format: `custom_id`, `method`, `url`, `body` | [Batch → Prepare input](https://developers.openai.com/api/docs/guides/batch#1-prepare-your-batch-file) | Same request-envelope fields; changed URL to `/v1/responses` |
| `client.files.create(...)`, `client.batches.create(...)`, `client.batches.retrieve(...)`, `client.files.content(...)` in the **one-request test** | [Batch guide → Prepare / Upload / Create / Check / Retrieve](https://developers.openai.com/api/docs/guides/batch) | **Same official API methods** as the full Batch; the one-line JSONL file, test flags, custom prompt, and result-to-JSONL conversion are my additions |
| `client.files.create(..., purpose="batch")` | [Batch → Upload input](https://developers.openai.com/api/docs/guides/batch#2-upload-your-batch-input-file) | Nearly verbatim official Python example |
| `client.batches.create(input_file_id=..., endpoint=..., completion_window="24h")` | [Batch → Create](https://developers.openai.com/api/docs/guides/batch#3-create-the-batch) | Same method; endpoint changed to `/v1/responses`, optional metadata omitted |
| `client.batches.retrieve(...)` | [Batch → Check status](https://developers.openai.com/api/docs/guides/batch#4-check-the-status-of-a-batch) | Same API method |
| `client.files.content(...)` | [Batch → Retrieve results](https://developers.openai.com/api/docs/guides/batch#5-retrieve-the-results) | Same API method; `.text` reads the downloaded JSONL result |
| Python `json.loads`, `json.dumps`, loops, example-count handling and result extraction | [Python JSON library](https://docs.python.org/3/library/json.html); [Responses response structure](https://developers.openai.com/api/reference/resources/responses/methods/create) | **Custom glue code** (not a verbatim OpenAI code example). Needed to make requests, validate output, and build one JSONL dataset |
| `Path.cwd()`, `Path(...).resolve()`, `getpass(...)` | [Python pathlib](https://docs.python.org/3/library/pathlib.html); [Python getpass](https://docs.python.org/3/library/getpass.html) | Local file locations and hidden API key input (not OpenAI API) |